In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent


In [ ]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load() 

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)


In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

In [ ]:
vector_store = Chroma.from_documents(documents=splitted_docs,embedding=embedding)

In [ ]:
@tool
def retriver_tool(query:str):
    """ 
        This tool can help you to retrieve the relevant data of pdf documents, and these documents have
        details about medical reports.
    """
    docs = vector_store.similarity_search(query=query,k=4)
    print("tool called",query)
    context = ""
    for doc in docs:
        context+=doc.page_content + "\n\n"
    return context

In [ ]:
llm = ChatGroq(
    groq_api_key=os.getenv("groq_api"), 
    model="openai/gpt-oss-20b"
)

In [ ]:
system_prompt = """
    You are a helpful assistant that answers questions using retrived context.
    ALWAYS use the `retriever_tool` tool for questions requiring external knowledge. 
"""

In [ ]:
agent = create_agent(model=llm,tools=[retriver_tool],system_prompt=system_prompt)

In [ ]:
query = "what is the name of the patient, and the doctors?"
res = agent.invoke({"messages":[{"role":"user","content":query}]})

In [ ]:
print(res["messages"][-1].content)